# 00 — Load & Preprocessing Data Evaluasi

Notebook ini adalah **entry point pipeline**. Tugasnya:
1. Load semua CSV hasil evaluasi (top-1 s/d top-10)
2. Bersihkan tipe data (N/A → NaN, string → float)
3. Klasifikasi query: `tabel` vs `narasi`
4. Simpan hasil ke `data_eval.pkl` untuk dipakai notebook selanjutnya

> Jalankan notebook ini **pertama kali** sebelum notebook visualisasi lainnya.

## 1. Import & Path Setup

In [1]:
import glob
import json
import pickle
from pathlib import Path

import pandas as pd

ROOT    = Path("../")
GEN_DIR = ROOT / "results" / "final" / "generation"
FIG_DIR = ROOT / "results" / "final" / "figures"
PKL_DIR = ROOT / "notebooks" / "_cache"

FIG_DIR.mkdir(parents=True, exist_ok=True)
PKL_DIR.mkdir(parents=True, exist_ok=True)

print(f"GEN_DIR : {GEN_DIR.resolve()}")
print(f"FIG_DIR : {FIG_DIR.resolve()}")
print(f"PKL_DIR : {PKL_DIR.resolve()}")

GEN_DIR : D:\My Files\Kuliah\SKRIPSI\rag-skripsi\results\final\generation
FIG_DIR : D:\My Files\Kuliah\SKRIPSI\rag-skripsi\results\final\figures
PKL_DIR : D:\My Files\Kuliah\SKRIPSI\rag-skripsi\notebooks\_cache


## 2. Load CSV Hasil Evaluasi (Top-1 s/d Top-10)

In [2]:
csv_files = sorted(GEN_DIR.glob("eval_*.csv"))
print(f"Ditemukan {len(csv_files)} file CSV:")
for f in csv_files:
    print(f"  {f.name}")

Ditemukan 10 file CSV:
  eval_20260531_181551_full_top1.csv
  eval_20260531_182241_full_top2.csv
  eval_20260531_183044_full_top3.csv
  eval_20260531_183822_full_top4.csv
  eval_20260531_184708_full_top5.csv
  eval_20260531_185558_full_top6.csv
  eval_20260531_190431_full_top7.csv
  eval_20260531_191252_full_top8.csv
  eval_20260531_192113_full_top9.csv
  eval_20260531_192940_full_top10.csv


In [3]:
dfs = []
for f in csv_files:
    topk = int(f.stem.split("_top")[1])
    df   = pd.read_csv(f)
    df["top_k"] = topk
    dfs.append(df)

all_df = pd.concat(dfs, ignore_index=True)
print(f"Shape gabungan: {all_df.shape}")
all_df.head(3)

Shape gabungan: (900, 13)


,query_id,method,question,gold_answer,generated_answer,precision_at_k,recall_at_k,mrr,bleu,rouge_l_recall,error,hardware_info,top_k
0,Q001,Element-Based,Berapa nilai Benchmark Indeks Hari-Orang Peker...,Benchmark Indeks Hari-Orang Pekerja Tidak Teta...,Tidak ada informasi dalam konteks yang menyebu...,0.0,0.0,0.0,0.1411,0.6190,NaN,"{""cpu"": ""x86_64"", ""cpu_count"": 64, ""cpu_count_...",1
1,Q001,MaxMin Semantic,Berapa nilai Benchmark Indeks Hari-Orang Peker...,Benchmark Indeks Hari-Orang Pekerja Tidak Teta...,Tidak memiliki informasi yang memadai untuk me...,0.0,0.0,0.0,0.1227,0.4286,NaN,"{""cpu"": ""x86_64"", ""cpu_count"": 64, ""cpu_count_...",1
2,Q001,Recursive,Berapa nilai Benchmark Indeks Hari-Orang Peker...,Benchmark Indeks Hari-Orang Pekerja Tidak Teta...,Nilai Benchmark Indeks Hari-Orang Pekerja Tida...,0.0,0.0,0.0,0.1374,0.6667,NaN,"{""cpu"": ""x86_64"", ""cpu_count"": 64, ""cpu_count_...",1


## 3. Bersihkan Tipe Data

In [4]:
# Kolom retrieval bisa berisi string 'N/A' untuk query tanpa GT chunk → konversi ke NaN
for col in ["precision_at_k", "recall_at_k", "mrr"]:
    all_df[col] = pd.to_numeric(all_df[col], errors="coerce")

# Verifikasi
print("Tipe data setelah konversi:")
print(all_df[["precision_at_k","recall_at_k","mrr","bleu","rouge_l_recall"]].dtypes)
print()
print(f"Baris dengan NaN di retrieval (query tanpa GT chunk): {all_df['mrr'].isna().sum()}")

Tipe data setelah konversi:
precision_at_k    float64
recall_at_k       float64
mrr               float64
bleu              float64
rouge_l_recall    float64
dtype: object

Baris dengan NaN di retrieval (query tanpa GT chunk): 40


## 4. Klasifikasi Query: Tabel vs Narasi

In [5]:
qa_path = ROOT / "data" / "ground_truth" / "qa_pairs_binary.json"
with open(qa_path, encoding="utf-8") as f:
    qa_list = json.load(f)

NARASI_KW = [
    "apa yang dimaksud", "jelaskan", "bagaimana",
    "ke dalam kelompok", "apa saja", "apa yang",
]

def classify_query(question: str) -> str:
    ql = question.lower()
    return "narasi" if any(kw in ql for kw in NARASI_KW) else "tabel"

qa_type = {item["id"]: classify_query(item["question"]) for item in qa_list}
all_df["query_type"] = all_df["query_id"].map(qa_type)

type_counts = all_df[all_df["top_k"] == 5].groupby(["query_type","method"]).size().unstack()
print("Distribusi tipe query (top-5 saja, per baris unik):")
print(f"  Tabel  : {(all_df[all_df['top_k']==5]['query_type']=='tabel').sum() // 3} query")
print(f"  Narasi : {(all_df[all_df['top_k']==5]['query_type']=='narasi').sum() // 3} query")

Distribusi tipe query (top-5 saja, per baris unik):
  Tabel  : 25 query
  Narasi : 5 query


## 5. Cek Statistik Dasar

In [6]:
top5 = all_df[all_df["top_k"] == 5].copy()

summary = (
    top5
    .groupby("method")[["precision_at_k","recall_at_k","mrr","bleu","rouge_l_recall"]]
    .mean()
    .round(4)
)
print("=== Rata-rata metrik per metode (Top-5) ===")
display(summary)

=== Rata-rata metrik per metode (Top-5) ===


,precision_at_k,recall_at_k,mrr,bleu,rouge_l_recall
method,,,,,
Element-Based,0.1286,0.3923,0.3542,0.1178,0.7005
MaxMin Semantic,0.1310,0.4466,0.3488,0.1052,0.6369
Recursive,0.1448,0.3523,0.4293,0.1440,0.6838


## 6. Simpan Cache untuk Notebook Selanjutnya

In [7]:
cache = {
    "all_df":   all_df,
    "top5":     top5,
    "qa_type":  qa_type,
    "FIG_DIR":  FIG_DIR,
}

cache_path = PKL_DIR / "data_eval.pkl"
with open(cache_path, "wb") as f:
    pickle.dump(cache, f)

print(f"Cache disimpan ke: {cache_path.resolve()}")
print("Siap untuk notebook visualisasi 01–06.")

Cache disimpan ke: D:\My Files\Kuliah\SKRIPSI\rag-skripsi\notebooks\_cache\data_eval.pkl
Siap untuk notebook visualisasi 01–06.
